# Intercropping DNA: 128-D monocrop-signature admixture through w1 → w4

This notebook answers a specific question: **how much of an intercropped pixel's fitted TESSERA embedding signature is attributable to each monocrop reference?**

For each raw 128-dimensional pixel embedding, it solves the constrained decomposition

\[
z_{pixel} \approx \sum_c w_c p_c,
\qquad w_c \ge 0,
\qquad \sum_c w_c = 1,
\]

where the four monocrop prototypes are Maize, Bean, Irish Potato, and Rice. It also solves a second, named-parent-only decomposition for Bean–Maize or Irish-Potato–Maize.

The unrestricted four-crop weights are the primary raw DNA-style result and are always shown. The two-parent ratio is interpreted only beside named-parent mass, off-target mass, whether the projection falls inside the parent segment, and a scaled monocrop-control residual percentile. Grid-block OOF controls are used when feasible; the in-sample fallback prevents a validated detection but does not hide the raw decomposition. These are **embedding-signature shares—not planted-area, plant-count, biomass, or yield percentages**.

Every clean field for the six requested labels and the exact same physical pixels are used in all four cumulative windows; geography is diagnostic only and does not subsample references. One affine metric is fitted from monocrop references pooled over w1–w3 and then held fixed; w4 is an out-of-contract sensitivity window.

In [ ]:
from functools import reduce
from pathlib import Path
import os
import sys

from IPython import get_ipython
from IPython.display import Markdown, display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely import wkt as shapely_wkt
from shapely.ops import transform as shapely_transform

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
notebook_dir = repo_root / "plain_tessera_incremental" / "notebooks"
for path in (repo_root, notebook_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _admixture import (
    dual_parent_evidence,
    solve_parent_segment,
    solve_simplex_admixture,
)
from _pixel_analysis import (
    clean_window_rows,
    load_embeddings,
    load_static,
    scan_window,
)

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
)
WINDOWS = ("w1", "w2", "w3", "w4")
FIT_WINDOWS = ("w1", "w2", "w3")
MONO_CROPS = ("Maize", "Bean", "Irish Potato", "Rice")
MIXTURE_PARENTS = {
    "Bean and Maize": ("Bean", "Maize"),
    "Irish Potato and Maize": ("Irish Potato", "Maize"),
}
ALL_LABELS = (*MONO_CROPS, *MIXTURE_PARENTS)
CROP_COLORS = {
    "Maize": "#E3A51A",
    "Bean": "#2E8B57",
    "Irish Potato": "#8E63B0",
    "Rice": "#3B82C4",
}
PIXEL_MAP_FIELDS_PER_MIXTURE = 2
SPATIAL_BLOCK_M = 5000
MAX_FOLDS = 5
SPATIAL_ASSIGNMENT_ATTEMPTS = 5000
MIN_VALIDATION_BLOCKS_PER_CROP = 3
MIN_BALANCED_ACCURACY = 0.50
MAX_ENDPOINT_MAE = 0.35
MIN_ENDPOINT_CORRECT_SIDE = 0.70
MIN_NAMED_PARENT_MASS = 0.50
MAX_PROTOTYPE_CONDITION = 50.0
N_REFERENCE_BOOTSTRAPS = 200
RANDOM_SEED = 20260708

pipeline_complete = (OUTPUT_DIR / "COMPLETED.json").is_file()
RUN_STATUS = (
    "COMPLETE PIPELINE"
    if pipeline_complete
    else "PARTIAL PIPELINE SNAPSHOT — DESCRIPTIVE ONLY"
)
print("Output:", OUTPUT_DIR)
print("Run status:", RUN_STATUS)

In [ ]:
def haversine_matrix_km(left_lon, left_lat, right_lon, right_lat):
    left_lon = np.deg2rad(np.asarray(left_lon, dtype=np.float64))[:, None]
    left_lat = np.deg2rad(np.asarray(left_lat, dtype=np.float64))[:, None]
    right_lon = np.deg2rad(np.asarray(right_lon, dtype=np.float64))[None, :]
    right_lat = np.deg2rad(np.asarray(right_lat, dtype=np.float64))[None, :]
    delta_lon = right_lon - left_lon
    delta_lat = right_lat - left_lat
    value = (
        np.sin(delta_lat / 2.0) ** 2
        + np.cos(left_lat) * np.cos(right_lat) * np.sin(delta_lon / 2.0) ** 2
    )
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(value, 0.0, 1.0)))


static = load_static(OUTPUT_DIR)
window_rows = []
for window in WINDOWS:
    spec = static.window_specs[window]
    start = pd.Timestamp(str(spec["start"]))
    end = pd.Timestamp(str(spec["end_exclusive"]))
    duration = int((end - start).days)
    window_rows.append(
        {
            "window_id": window,
            "start": start.date().isoformat(),
            "end_exclusive": end.date().isoformat(),
            "duration_days": duration,
            "contract_status": (
                "within annual contract"
                if duration <= 366
                else "OOD sensitivity; repeated day-of-year values"
            ),
        }
    )
window_table = pd.DataFrame(window_rows).set_index("window_id")

candidate_field_uids = frozenset(
    static.fields.loc[
        static.fields["landcover"].isin(ALL_LABELS), "field_uid"
    ].astype(str)
)
if not candidate_field_uids:
    raise RuntimeError("None of the requested crop labels exists in fields.parquet.")

scans = {
    window: scan_window(
        static,
        window,
        retained_field_uids=candidate_field_uids,
    )
    for window in WINDOWS
}
common_tasks = reduce(
    set.intersection,
    (set(scan.task_fingerprints) for scan in scans.values()),
)
for task_key in common_tasks:
    fingerprints = {scans[w].task_fingerprints[task_key] for w in WINDOWS}
    if len(fingerprints) != 1:
        raise RuntimeError(f"Task fingerprint changes across windows: {task_key}")

common_published_fields = candidate_field_uids.intersection(
    *(set(scan.published_fields) for scan in scans.values())
)
if not common_published_fields:
    raise RuntimeError("No candidate field is fully published in every window yet.")
clean_by_window = {
    window: clean_window_rows(
        static,
        scans[window],
        common_published_fields,
    )
    for window in WINDOWS
}
pair_sets = {
    window: set(
        clean.rows[["field_uid", "pixel_id"]].itertuples(
            index=False,
            name=None,
        )
    )
    for window, clean in clean_by_window.items()
}
common_pairs = reduce(set.intersection, pair_sets.values())
if not common_pairs:
    raise RuntimeError("No complete clean physical pixels are shared by w1–w4.")
common_pair_index = pd.MultiIndex.from_tuples(
    sorted(common_pairs),
    names=["field_uid", "pixel_id"],
)
base_rows = clean_by_window["w4"].rows.set_index(["field_uid", "pixel_id"])
base_rows = base_rows.loc[
    base_rows.index.intersection(common_pair_index)
].reset_index()

mixture_index = (
    base_rows.loc[
        base_rows["landcover"].isin(MIXTURE_PARENTS),
        ["field_uid", "pixel_id", "landcover"],
    ]
    .drop_duplicates()
    .sort_values(["landcover", "field_uid", "pixel_id"], kind="stable")
)
missing_mixtures = set(MIXTURE_PARENTS).difference(mixture_index["landcover"])
if missing_mixtures:
    raise RuntimeError(f"Missing intercropping labels: {sorted(missing_mixtures)}")
all_label_summary = (
    base_rows[base_rows["landcover"].isin(ALL_LABELS)].groupby("landcover")
    .agg(
        fields=("field_uid", "nunique"),
        pixel_samples=("pixel_id", "nunique"),
    )
    .sort_values(["fields", "pixel_samples"], ascending=False)
)
mono_candidate_fields = set(
    base_rows.loc[
        base_rows["landcover"].isin(MONO_CROPS), "field_uid"
    ].astype(str)
)

spatial_records = []
spatial_memberships = static.memberships[
    static.memberships["field_uid"].isin(
        set(mixture_index["field_uid"]) | mono_candidate_fields
    )
]
for field_uid, rows in spatial_memberships.groupby("field_uid", sort=True):
    epsg_values = rows["utm_epsg"].astype(np.int64).unique()
    if len(epsg_values) != 1:
        raise RuntimeError(f"Field {field_uid} spans multiple UTM grids.")
    epsg = int(epsg_values[0])
    centroid_x_m = float(
        ((rows["pixel_x_index"].to_numpy(np.float64) + 0.5) * 10.0).mean()
    )
    centroid_y_m = float(
        ((rows["pixel_y_index"].to_numpy(np.float64) + 0.5) * 10.0).mean()
    )
    spatial_records.append(
        {
            "field_uid": str(field_uid),
            "landcover": str(static.field_lookup.loc[str(field_uid), "landcover"]),
            "utm_epsg": epsg,
            "centroid_longitude": float(rows["pixel_longitude"].mean()),
            "centroid_latitude": float(rows["pixel_latitude"].mean()),
            "spatial_block": (
                f"epsg{epsg}-x{int(np.floor(centroid_x_m / SPATIAL_BLOCK_M))}"
                f"-y{int(np.floor(centroid_y_m / SPATIAL_BLOCK_M))}"
            ),
        }
    )
field_spatial = pd.DataFrame(spatial_records)
if field_spatial["field_uid"].duplicated().any():
    raise RuntimeError("Spatial metadata contains duplicate fields.")

reference_panels = {}
match_records = []
selected_reference_fields = set()
for mixture_label in MIXTURE_PARENTS:
    mixture_field_ids = sorted(
        mixture_index.loc[
            mixture_index["landcover"].eq(mixture_label), "field_uid"
        ].astype(str).unique()
    )
    mixture_geo = field_spatial[
        field_spatial["field_uid"].isin(mixture_field_ids)
    ]
    panel = {}
    for crop in MONO_CROPS:
        candidates = field_spatial[
            field_spatial["field_uid"].isin(mono_candidate_fields)
            & field_spatial["landcover"].eq(crop)
        ].copy()
        if candidates.empty:
            raise RuntimeError(f"No common w1–w4 monocrop candidates for {crop}.")
        distances = haversine_matrix_km(
            candidates["centroid_longitude"],
            candidates["centroid_latitude"],
            mixture_geo["centroid_longitude"],
            mixture_geo["centroid_latitude"],
        )
        candidates["distance_to_mixture_km"] = distances.min(axis=1)
        selected = candidates.sort_values(
            ["distance_to_mixture_km", "field_uid"],
            kind="stable",
        )
        field_ids = selected["field_uid"].astype(str).tolist()
        if len(field_ids) < 2:
            raise RuntimeError(f"Fewer than two reference fields for {crop}.")
        panel[crop] = field_ids
        selected_reference_fields.update(field_ids)
        mixture_to_reference = haversine_matrix_km(
            mixture_geo["centroid_longitude"],
            mixture_geo["centroid_latitude"],
            selected["centroid_longitude"],
            selected["centroid_latitude"],
        ).min(axis=1)
        match_records.append(
            {
                "mixture": mixture_label,
                "reference_crop": crop,
                "reference_fields": len(field_ids),
                "mixture_fields": len(mixture_field_ids),
                "median_mixture_to_reference_km": float(
                    np.median(mixture_to_reference)
                ),
                "max_mixture_to_reference_km": float(
                    np.max(mixture_to_reference)
                ),
            }
        )
    reference_panels[mixture_label] = panel

selected_field_ids = selected_reference_fields | set(
    mixture_index["field_uid"].astype(str)
)
selected_base = base_rows[
    base_rows["field_uid"].isin(selected_field_ids)
].copy()
if selected_base.duplicated(["field_uid", "pixel_id"]).any():
    raise RuntimeError("Selected cohort contains duplicate field/pixel rows.")

diagnostics = pd.concat(
    {window: clean_by_window[window].diagnostics for window in WINDOWS},
    axis=1,
)
display(window_table)
display(all_label_summary)
display(pd.DataFrame(match_records).set_index(["mixture", "reference_crop"]))
display(diagnostics)
print(
    f"Frozen common cohort: {selected_base['field_uid'].nunique()} fields, "
    f"{selected_base['pixel_id'].nunique()} exact physical pixels."
)

In [ ]:
selected_pairs = selected_base[["field_uid", "pixel_id"]].copy()
timeline_parts = []
for window in WINDOWS:
    selected = selected_pairs.merge(
        clean_by_window[window].rows,
        on=["field_uid", "pixel_id"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    if not selected["_merge"].eq("both").all():
        raise RuntimeError(f"A frozen cohort pixel disappeared from {window}.")
    loaded = load_embeddings(
        static,
        scans[window],
        selected.drop(columns="_merge"),
    )
    loaded["window_id"] = window
    loaded["window_order"] = WINDOWS.index(window)
    loaded["pixel_x_index"] = loaded["pixel_id"].map(
        static.pixel_lookup["pixel_x_index"]
    ).astype(np.int64)
    loaded["pixel_y_index"] = loaded["pixel_id"].map(
        static.pixel_lookup["pixel_y_index"]
    ).astype(np.int64)
    loaded["utm_epsg"] = loaded["pixel_id"].map(
        static.pixel_lookup["utm_epsg"]
    ).astype(np.int64)
    loaded["_raw"] = [
        np.asarray(vector, dtype=np.float64) for vector in loaded["_vector"]
    ]
    timeline_parts.append(loaded)
timeline = pd.concat(timeline_parts, ignore_index=True)
timeline = timeline.sort_values(
    ["field_uid", "pixel_id", "window_order"],
    kind="stable",
).reset_index(drop=True)
if len(timeline) != len(selected_pairs) * len(WINDOWS):
    raise RuntimeError("Frozen four-window timeline has missing rows.")
if timeline.duplicated(["field_uid", "pixel_id", "window_id"]).any():
    raise RuntimeError("Frozen four-window timeline contains duplicate rows.")

count_columns = [
    "s2_source_count", "s1_source_count",
    "s2_valid_count", "s1_valid_count",
    "s2_input_count", "s1_input_count",
]
count_deltas = timeline.groupby(["field_uid", "pixel_id"])[
    count_columns
].diff()
if (count_deltas.dropna().to_numpy(np.float64) < 0).any():
    raise RuntimeError("A cumulative source/input count decreases across windows.")


def reference_row_weights(frame):
    fields = frame[["window_id", "landcover", "field_uid"]].drop_duplicates()
    fields_per_group = fields.groupby(["window_id", "landcover"]).size().to_dict()
    pixels_per_field = frame.groupby(
        ["window_id", "landcover", "field_uid"]
    ).size().to_dict()
    window_count = frame["window_id"].nunique()
    crop_count = frame["landcover"].nunique()
    weights = []
    for window, crop, field_uid in frame[
        ["window_id", "landcover", "field_uid"]
    ].itertuples(index=False, name=None):
        weights.append(
            1.0
            / (
                window_count
                * crop_count
                * fields_per_group[(window, crop)]
                * pixels_per_field[(window, crop, field_uid)]
            )
        )
    return np.asarray(weights, dtype=np.float64)


def fit_standardizer(reference_rows):
    fit = reference_rows[reference_rows["window_id"].isin(FIT_WINDOWS)].copy()
    if set(fit["landcover"].unique()) != set(MONO_CROPS):
        raise RuntimeError("Standardizer fit lacks one or more monocrop classes.")
    matrix = np.stack(fit["_raw"].to_numpy())
    weights = reference_row_weights(fit)
    mean = np.average(matrix, axis=0, weights=weights)
    variance = np.average((matrix - mean) ** 2, axis=0, weights=weights)
    raw_scale = np.sqrt(np.maximum(variance, 0.0))
    positive = raw_scale[raw_scale > 0]
    floor = max(float(np.median(positive)) * 0.05, 1e-6)
    scale = np.maximum(raw_scale, floor)
    return {"mean": mean, "scale": scale, "scale_floor": floor}


def transform_rows(frame, standardizer):
    matrix = np.stack(frame["_raw"].to_numpy())
    return (matrix - standardizer["mean"]) / standardizer["scale"]


def fit_window_prototypes(reference_rows, standardizer, window):
    rows = reference_rows[reference_rows["window_id"].eq(window)].copy()
    transformed = transform_rows(rows, standardizer)
    field_records = []
    for (field_uid, crop), positions in rows.groupby(
        ["field_uid", "landcover"],
        sort=True,
    ).indices.items():
        field_records.append(
            {
                "field_uid": str(field_uid),
                "landcover": str(crop),
                "signature": transformed[np.asarray(positions)].mean(axis=0),
            }
        )
    field_signatures = pd.DataFrame(field_records)
    prototypes = []
    within_rms = []
    for crop in MONO_CROPS:
        crop_matrix = np.stack(
            field_signatures.loc[
                field_signatures["landcover"].eq(crop), "signature"
            ].to_numpy()
        )
        prototype = crop_matrix.mean(axis=0)
        prototypes.append(prototype)
        within_rms.append(
            float(np.sqrt(np.mean(np.sum((crop_matrix - prototype) ** 2, axis=1))))
        )
    return np.stack(prototypes), np.asarray(within_rms), field_signatures


def score_embeddings(frame, standardizer, prototypes, parent_names):
    transformed = transform_rows(frame, standardizer)
    simplex = solve_simplex_admixture(
        transformed,
        prototypes,
        prototype_names=MONO_CROPS,
    )
    parent_a, parent_b = parent_names
    a_index = MONO_CROPS.index(parent_a)
    b_index = MONO_CROPS.index(parent_b)
    segment = solve_parent_segment(
        transformed,
        prototypes[a_index],
        prototypes[b_index],
        parent_names=parent_names,
    )
    result = frame.copy()
    result["_standardized"] = list(transformed)
    result["_simplex_reconstruction"] = list(simplex.reconstruction)
    result["_simplex_residual_vector"] = list(simplex.residual)
    for crop_index, crop in enumerate(MONO_CROPS):
        result[f"weight_{crop.lower().replace(' ', '_')}"] = simplex.weights[
            :, crop_index
        ]
    named_mass = simplex.weights[:, a_index] + simplex.weights[:, b_index]
    result["named_parent_mass"] = named_mass
    result["off_target_mass"] = 1.0 - named_mass
    result["simplex_parent_a_share"] = np.divide(
        simplex.weights[:, a_index],
        named_mass,
        out=np.full(len(result), np.nan),
        where=named_mass > 1e-12,
    )
    result["pair_parent_a_share"] = segment.weights[:, 0]
    result["pair_raw_parent_a_share"] = segment.raw_weights[:, 0]
    result["pair_in_segment"] = segment.in_segment
    result["minor_parent_share"] = np.minimum(
        result["pair_parent_a_share"],
        1.0 - result["pair_parent_a_share"],
    )
    result["dual_parent_score"] = dual_parent_evidence(
        simplex.weights,
        a_index,
        b_index,
    )
    result["share_disagreement"] = np.abs(
        result["simplex_parent_a_share"] - result["pair_parent_a_share"]
    )
    result["all_parent_residual"] = simplex.residual_norm
    result["pair_residual"] = segment.residual_norm
    result["simplex_active_crops"] = simplex.active_count
    result["parent_a"] = parent_a
    result["parent_b"] = parent_b
    return result


def equal_field_residual_radius(scored, residual_column, crops):
    rows = scored[scored["landcover"].isin(crops)]
    field_rms = (
        rows.assign(_squared=rows[residual_column] ** 2)
        .groupby(["landcover", "field_uid"])["_squared"]
        .mean()
        .pow(0.5)
        .rename("field_rms")
        .reset_index()
    )
    crop_mean_squared = field_rms.groupby("landcover")["field_rms"].apply(
        lambda values: float(np.mean(values.to_numpy() ** 2))
    )
    return max(float(np.sqrt(crop_mean_squared.mean())), 1e-12)


full_models = {}
model_records = []
for mixture_label, parents in MIXTURE_PARENTS.items():
    reference_ids = {
        field_uid
        for crop_ids in reference_panels[mixture_label].values()
        for field_uid in crop_ids
    }
    reference_rows = timeline[
        timeline["field_uid"].isin(reference_ids)
        & timeline["landcover"].isin(MONO_CROPS)
    ].copy()
    standardizer = fit_standardizer(reference_rows)
    for window in WINDOWS:
        prototypes, within_rms, field_signatures = fit_window_prototypes(
            reference_rows,
            standardizer,
            window,
        )
        parent_indices = [MONO_CROPS.index(parent) for parent in parents]
        pair_distance = float(
            np.linalg.norm(
                prototypes[parent_indices[0]] - prototypes[parent_indices[1]]
            )
        )
        pooled_within = float(
            np.sqrt(np.mean(within_rms[parent_indices] ** 2))
        )
        directions = prototypes[1:] - prototypes[0]
        singular_values = np.linalg.svd(directions, compute_uv=False)
        condition = (
            float(singular_values[0] / singular_values[-1])
            if singular_values[-1] > 1e-12
            else np.inf
        )
        reference_window = reference_rows[
            reference_rows["window_id"].eq(window)
        ]
        reference_scored = score_embeddings(
            reference_window,
            standardizer,
            prototypes,
            parents,
        )
        parent_reference = reference_scored[
            reference_scored["landcover"].isin(parents)
        ]
        residual_scale = equal_field_residual_radius(
            reference_scored, "all_parent_residual", MONO_CROPS
        )
        pair_residual_scale = equal_field_residual_radius(
            parent_reference, "pair_residual", parents
        )
        full_models[(mixture_label, window)] = {
            "standardizer": standardizer,
            "prototypes": prototypes,
            "within_rms": within_rms,
            "field_signatures": field_signatures,
            "pair_separation": pair_distance / max(pooled_within, 1e-12),
            "condition_number": condition,
            "residual_scale": residual_scale,
            "pair_residual_scale": pair_residual_scale,
        }
        model_records.append(
            {
                "mixture": mixture_label,
                "window_id": window,
                "pair_separation": pair_distance / max(pooled_within, 1e-12),
                "prototype_condition_number": condition,
                "prototype_condition_passed": bool(
                    np.isfinite(condition)
                    and condition <= MAX_PROTOTYPE_CONDITION
                ),
                "standardizer_scale_floor": standardizer["scale_floor"],
            }
        )
model_table = pd.DataFrame(model_records)
display(model_table.set_index(["mixture", "window_id"]))
print(
    f"Loaded {len(timeline):,} raw 128-D rows = "
    f"{selected_pairs['pixel_id'].nunique():,} pixels × {len(WINDOWS)} windows."
)

In [ ]:
def assign_spatial_folds(field_table, seed):
    block_counts = (
        field_table.groupby("landcover")["spatial_block"]
        .nunique()
        .reindex(MONO_CROPS, fill_value=0)
    )
    if (block_counts < 2).any():
        return None, 0, block_counts
    n_folds = min(MAX_FOLDS, int(block_counts.min()))
    blocks = np.array(sorted(field_table["spatial_block"].unique()), dtype=object)
    crop_index = {crop: index for index, crop in enumerate(MONO_CROPS)}
    block_index = {block: index for index, block in enumerate(blocks)}
    counts = np.zeros((len(blocks), len(MONO_CROPS)), dtype=np.int64)
    for (block, crop), count in field_table.groupby(
        ["spatial_block", "landcover"]
    ).size().items():
        counts[block_index[block], crop_index[crop]] = int(count)
    rng = np.random.default_rng(seed)
    target = counts.sum(axis=0) / n_folds
    best = None
    best_score = np.inf
    for _ in range(SPATIAL_ASSIGNMENT_ATTEMPTS):
        assignment = rng.integers(0, n_folds, size=len(blocks))
        if len(np.unique(assignment)) != n_folds:
            continue
        fold_counts = np.stack(
            [counts[assignment == fold].sum(axis=0) for fold in range(n_folds)]
        )
        if (fold_counts == 0).any():
            continue
        score = float((((fold_counts - target) ** 2) / (target + 1e-9)).sum())
        if score < best_score:
            best = assignment.copy()
            best_score = score
    if best is None:
        return None, 0, block_counts
    return dict(zip(blocks, best, strict=True)), n_folds, block_counts


oof_pixel_parts = []
oof_field_parts = []
validation_records = []
validation_matrices = {}
residual_controls = {}
pair_residual_controls = {}
field_residual_controls = {}
field_pair_residual_controls = {}
named_mass_thresholds = {}
pixel_dual_thresholds = {}
field_dual_thresholds = {}
validation_gates = {}

for cohort_index, (mixture_label, parents) in enumerate(MIXTURE_PARENTS.items()):
    reference_ids = {
        field_uid
        for crop_ids in reference_panels[mixture_label].values()
        for field_uid in crop_ids
    }
    reference_meta = field_spatial[
        field_spatial["field_uid"].isin(reference_ids)
    ][["field_uid", "landcover", "spatial_block"]].drop_duplicates()
    block_to_fold, n_folds, block_counts = assign_spatial_folds(
        reference_meta,
        RANDOM_SEED + 100 + cohort_index,
    )
    spatial_feasible = block_to_fold is not None
    all_reference_rows = timeline[
        timeline["field_uid"].isin(reference_ids)
        & timeline["landcover"].isin(MONO_CROPS)
    ].copy()
    cohort_parts = []
    if spatial_feasible:
        reference_meta = reference_meta.copy()
        reference_meta["fold"] = reference_meta["spatial_block"].map(
            block_to_fold
        ).astype(int)
        for fold in range(n_folds):
            train_ids = set(
                reference_meta.loc[reference_meta["fold"].ne(fold), "field_uid"]
            )
            test_ids = set(
                reference_meta.loc[reference_meta["fold"].eq(fold), "field_uid"]
            )
            if train_ids & test_ids:
                raise RuntimeError("A spatial validation field leaked across folds.")
            train_rows = all_reference_rows[
                all_reference_rows["field_uid"].isin(train_ids)
            ]
            standardizer = fit_standardizer(train_rows)
            for window in WINDOWS:
                prototypes, _, _ = fit_window_prototypes(
                    train_rows,
                    standardizer,
                    window,
                )
                train_window = train_rows[train_rows["window_id"].eq(window)]
                train_scored = score_embeddings(
                    train_window,
                    standardizer,
                    prototypes,
                    parents,
                )
                train_parent = train_scored[
                    train_scored["landcover"].isin(parents)
                ]
                residual_scale = equal_field_residual_radius(
                    train_scored,
                    "all_parent_residual",
                    MONO_CROPS,
                )
                pair_scale = equal_field_residual_radius(
                    train_parent,
                    "pair_residual",
                    parents,
                )
                test_rows = all_reference_rows[
                    all_reference_rows["field_uid"].isin(test_ids)
                    & all_reference_rows["window_id"].eq(window)
                ]
                scored = score_embeddings(
                    test_rows,
                    standardizer,
                    prototypes,
                    parents,
                )
                scored["all_parent_residual_scaled"] = (
                    scored["all_parent_residual"] / residual_scale
                )
                scored["pair_residual_scaled"] = scored["pair_residual"] / pair_scale
                scored["mixture_cohort"] = mixture_label
                scored["control_source"] = "grid_block_oof"
                cohort_parts.append(scored)
    else:
        for window in WINDOWS:
            model = full_models[(mixture_label, window)]
            test_rows = all_reference_rows[
                all_reference_rows["window_id"].eq(window)
            ]
            scored = score_embeddings(
                test_rows,
                model["standardizer"],
                model["prototypes"],
                parents,
            )
            scored["all_parent_residual_scaled"] = (
                scored["all_parent_residual"] / model["residual_scale"]
            )
            scored["pair_residual_scaled"] = (
                scored["pair_residual"] / model["pair_residual_scale"]
            )
            scored["mixture_cohort"] = mixture_label
            scored["control_source"] = "in_sample_fallback"
            cohort_parts.append(scored)

    cohort_pixels = pd.concat(cohort_parts, ignore_index=True)
    oof_pixel_parts.append(cohort_pixels)
    weight_columns = [
        f"weight_{crop.lower().replace(' ', '_')}" for crop in MONO_CROPS
    ]
    field_controls = cohort_pixels.groupby(
        ["mixture_cohort", "window_id", "field_uid", "landcover"],
        as_index=False,
    ).agg(
        **{column: (column, "mean") for column in weight_columns},
        pair_parent_a_share=("pair_parent_a_share", "mean"),
        named_parent_mass=("named_parent_mass", "mean"),
        off_target_mass=("off_target_mass", "mean"),
        dual_parent_score=("dual_parent_score", "mean"),
        all_parent_residual_scaled=("all_parent_residual_scaled", "median"),
        pair_residual_scaled=("pair_residual_scaled", "median"),
        pair_in_segment_rate=("pair_in_segment", "mean"),
    )
    parent_columns = [
        f"weight_{parent.lower().replace(' ', '_')}" for parent in parents
    ]
    field_controls["named_parent_mass"] = field_controls[
        parent_columns
    ].sum(axis=1)
    field_controls["unrestricted_parent_a_share"] = np.divide(
        field_controls[parent_columns[0]].to_numpy(np.float64),
        field_controls["named_parent_mass"].to_numpy(np.float64),
        out=np.full(len(field_controls), np.nan, dtype=np.float64),
        where=field_controls["named_parent_mass"].to_numpy() > 1e-12,
    )
    field_controls["dual_parent_score"] = (
        4.0
        * field_controls[parent_columns[0]]
        * field_controls[parent_columns[1]]
        / field_controls["named_parent_mass"].clip(lower=1e-12)
    )
    field_controls["control_source"] = (
        "grid_block_oof" if spatial_feasible else "in_sample_fallback"
    )
    oof_field_parts.append(field_controls)

    for window in WINDOWS:
        rows = field_controls[field_controls["window_id"].eq(window)].copy()
        true = rows["landcover"].astype(str).to_numpy()
        weights = rows[weight_columns].to_numpy()
        predicted = np.asarray(MONO_CROPS, dtype=object)[np.argmax(weights, axis=1)]
        recalls = []
        recovered = np.zeros((len(MONO_CROPS), len(MONO_CROPS)))
        for crop_index, crop in enumerate(MONO_CROPS):
            mask = true == crop
            recalls.append(float((predicted[mask] == crop).mean()))
            recovered[crop_index] = weights[mask].mean(axis=0)
        balanced_accuracy = float(np.mean(recalls))
        parent_mask = np.isin(true, parents)
        expected_a = (true[parent_mask] == parents[0]).astype(np.float64)
        estimated_a = rows.loc[
            parent_mask, "unrestricted_parent_a_share"
        ].to_numpy()
        endpoint_mae = float(np.mean(np.abs(estimated_a - expected_a)))
        correct_side = float(
            np.mean((estimated_a >= 0.5) == (expected_a >= 0.5))
        )
        window_pixels = cohort_pixels[cohort_pixels["window_id"].eq(window)]
        parent_pixels = window_pixels[
            window_pixels["landcover"].isin(parents)
        ]
        parent_fields = rows[rows["landcover"].isin(parents)]
        residual_controls[(mixture_label, window)] = np.sort(
            window_pixels["all_parent_residual_scaled"].to_numpy(np.float64)
        )
        pair_residual_controls[(mixture_label, window)] = np.sort(
            parent_pixels["pair_residual_scaled"].to_numpy(np.float64)
        )
        field_residual_controls[(mixture_label, window)] = np.sort(
            rows["all_parent_residual_scaled"].to_numpy(np.float64)
        )
        field_pair_residual_controls[(mixture_label, window)] = np.sort(
            parent_fields["pair_residual_scaled"].to_numpy(np.float64)
        )
        named_mass_thresholds[(mixture_label, window)] = max(
            float(parent_fields["named_parent_mass"].quantile(0.05)),
            MIN_NAMED_PARENT_MASS,
        )
        pixel_dual_thresholds[(mixture_label, window)] = float(
            window_pixels["dual_parent_score"].quantile(
                0.95,
                interpolation="higher",
            )
        )
        field_dual_thresholds[(mixture_label, window)] = float(
            rows["dual_parent_score"].quantile(
                0.95,
                interpolation="higher",
            )
        )
        condition = full_models[(mixture_label, window)]["condition_number"]
        condition_passed = bool(
            np.isfinite(condition) and condition <= MAX_PROTOTYPE_CONDITION
        )
        gate = bool(
            spatial_feasible
            and int(block_counts.min()) >= MIN_VALIDATION_BLOCKS_PER_CROP
            and len(rows) >= 20
            and balanced_accuracy >= MIN_BALANCED_ACCURACY
            and endpoint_mae <= MAX_ENDPOINT_MAE
            and correct_side >= MIN_ENDPOINT_CORRECT_SIDE
            and condition_passed
        )
        validation_gates[(mixture_label, window)] = gate
        validation_matrices[(mixture_label, window)] = recovered
        mono_dual_fpr = float(
            (rows["dual_parent_score"] > field_dual_thresholds[
                (mixture_label, window)
            ]).mean()
        )
        validation_records.append(
            {
                "mixture": mixture_label,
                "window_id": window,
                "control_source": (
                    "grid_block_oof" if spatial_feasible else "in_sample_fallback"
                ),
                "folds": n_folds,
                "minimum_grid_blocks_per_crop": int(block_counts.min()),
                "monocrop_null_fields": len(rows),
                "balanced_accuracy": balanced_accuracy,
                "endpoint_mae": endpoint_mae,
                "endpoint_correct_side_rate": correct_side,
                "scaled_residual_95pct": float(
                    np.quantile(
                        field_residual_controls[(mixture_label, window)],
                        0.95,
                    )
                ),
                "scaled_pair_residual_95pct": float(
                    np.quantile(
                        field_pair_residual_controls[
                            (mixture_label, window)
                        ],
                        0.95,
                    )
                ),
                "named_parent_mass_threshold": named_mass_thresholds[
                    (mixture_label, window)
                ],
                "dual_parent_field_threshold": field_dual_thresholds[
                    (mixture_label, window)
                ],
                "mono_parent_false_positive_rate": mono_dual_fpr,
                "pair_separation": full_models[
                    (mixture_label, window)
                ]["pair_separation"],
                "prototype_condition_number": condition,
                "prototype_condition_passed": condition_passed,
                "validation_gate_passed": gate,
            }
        )

oof_pixels = pd.concat(oof_pixel_parts, ignore_index=True)
oof_fields = pd.concat(oof_field_parts, ignore_index=True)
validation_table = pd.DataFrame(validation_records)
display(validation_table.set_index(["mixture", "window_id"]))

## How to read the decomposition tables

The notebook reports two separate layers:

1. **Raw descriptive decomposition — four-crop weights are always populated**
   - `weight_maize`, `weight_bean`, `weight_irish_potato`, and `weight_rice` are the unrestricted four-crop embedding weights and sum to one.
   - `unrestricted_parent_a_share` is parent A's share *within the two named-parent weights* from that four-crop fit. It is `NaN` only when `named_parent_mass == 0`, because a within-parent ratio is then mathematically undefined; the four crop weights remain available.
   - `pair_parent_a_share` is a separate forced projection using only the two named parents.
   - `named_parent_mass` is the combined unrestricted mass of the two named parents; `off_target_mass` is everything assigned to the other monocrops.
   - Field confidence intervals resample reference fields. Group confidence intervals jointly resample reference fields and mixture fields. These values remain visible even when validation fails.

2. **Validated detection — may be unavailable**
   - The 5 km grid is only a spatial cross-validation partition: neighboring field centroids in the same cell are held out together. It is not a field buffer, STAC query radius, pixel filter, or final-model subsample.
   - `validation_gate_passed` says whether monocrop controls recover known pure endpoints well enough.
   - `field_quality_passed` checks the individual field fit.
   - `detection_eligible` requires both checks. It does **not** control whether raw weights are displayed.
   - `dual_parent_score_exceeds_null` is a descriptive comparison with the monocrop null threshold.
   - `intercropping_detected` is `True` only when the field is detection-eligible and exceeds that threshold.

In the group table, `fields_total`, `fields_quality_passed`, `fields_detection_eligible`, and `fields_detected` make the denominator explicit. A `NaN` in `validated_detection_rate` means there were zero eligible fields; it does **not** mean embeddings or raw decompositions are missing.


In [ ]:
def empirical_percentile(values, controls):
    controls = np.asarray(controls, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    return 100.0 * np.searchsorted(controls, values, side="right") / max(
        len(controls),
        1,
    )


mixture_score_parts = []
for mixture_label, parents in MIXTURE_PARENTS.items():
    for window in WINDOWS:
        model = full_models[(mixture_label, window)]
        rows = timeline[
            timeline["landcover"].eq(mixture_label)
            & timeline["window_id"].eq(window)
        ]
        scored = score_embeddings(
            rows,
            model["standardizer"],
            model["prototypes"],
            parents,
        )
        scored["mixture_cohort"] = mixture_label
        scored["all_parent_residual_scaled"] = (
            scored["all_parent_residual"] / model["residual_scale"]
        )
        scored["pair_residual_scaled"] = (
            scored["pair_residual"] / model["pair_residual_scale"]
        )
        scored["residual_percentile"] = empirical_percentile(
            scored["all_parent_residual_scaled"],
            residual_controls[(mixture_label, window)],
        )
        scored["pair_residual_percentile"] = empirical_percentile(
            scored["pair_residual_scaled"],
            pair_residual_controls[(mixture_label, window)],
        )
        scored["named_mass_threshold"] = named_mass_thresholds[
            (mixture_label, window)
        ]
        scored["dual_parent_threshold"] = pixel_dual_thresholds[
            (mixture_label, window)
        ]
        scored["validation_gate_passed"] = validation_gates[
            (mixture_label, window)
        ]
        scored["fit_usable"] = (
            scored["pair_in_segment"]
            & scored["residual_percentile"].le(95.0)
            & scored["pair_residual_percentile"].le(95.0)
            & scored["named_parent_mass"].ge(scored["named_mass_threshold"])
        )
        scored["decomposition_usable"] = (
            scored["validation_gate_passed"] & scored["fit_usable"]
        )
        scored["dual_parent_evidence_positive"] = (
            scored["fit_usable"]
            & scored["dual_parent_score"].gt(scored["dual_parent_threshold"])
        )
        scored["dual_parent_detected"] = (
            scored["validation_gate_passed"]
            & scored["dual_parent_evidence_positive"]
        )
        for crop in MONO_CROPS:
            column = f"weight_{crop.lower().replace(' ', '_')}"
            scored[f"called_{column}"] = scored[column].where(
                scored["decomposition_usable"]
            )
        scored["called_pair_parent_a_share"] = scored[
            "pair_parent_a_share"
        ].where(scored["decomposition_usable"])
        scored["called_named_parent_mass"] = scored["named_parent_mass"].where(
            scored["decomposition_usable"]
        )
        scored["call_status"] = np.where(
            ~scored["validation_gate_passed"],
            "RAW ESTIMATE — VALIDATION FAILED",
            np.where(
                ~scored["fit_usable"],
                "RAW ESTIMATE — PIXEL QC FAILED",
                np.where(
                    scored["dual_parent_detected"],
                    "VALIDATED DUAL-PARENT DETECTION",
                    "VALIDATED DECOMPOSITION; NO DUAL-PARENT DETECTION",
                ),
            ),
        )
        mixture_score_parts.append(scored)
pixel_dna = pd.concat(mixture_score_parts, ignore_index=True)

weight_columns = [
    f"weight_{crop.lower().replace(' ', '_')}" for crop in MONO_CROPS
]
aggregate_spec = {}
for column in weight_columns:
    aggregate_spec[f"raw_{column}"] = (column, "mean")
    aggregate_spec[column] = (column, "mean")
field_dna = pixel_dna.groupby(
    ["mixture_cohort", "field_uid", "window_id", "window_order"],
    as_index=False,
).agg(
    **aggregate_spec,
    pixel_count=("pixel_id", "nunique"),
    raw_pair_parent_a_share=("pair_parent_a_share", "mean"),
    pair_parent_a_share=("pair_parent_a_share", "mean"),
    pair_parent_a_q10=(
        "pair_parent_a_share",
        lambda values: values.quantile(0.10),
    ),
    pair_parent_a_q90=(
        "pair_parent_a_share",
        lambda values: values.quantile(0.90),
    ),
    raw_named_parent_mass=("named_parent_mass", "mean"),
    named_parent_mass=("named_parent_mass", "mean"),
    raw_off_target_mass=("off_target_mass", "mean"),
    off_target_mass=("off_target_mass", "mean"),
    dual_parent_score=("dual_parent_score", "mean"),
    all_parent_residual_scaled=("all_parent_residual_scaled", "median"),
    pair_residual_scaled=("pair_residual_scaled", "median"),
    residual_percentile=("residual_percentile", "median"),
    pair_residual_percentile=("pair_residual_percentile", "median"),
    pair_in_segment_rate=("pair_in_segment", "mean"),
    share_disagreement=("share_disagreement", "mean"),
    usable_pixel_rate=("fit_usable", "mean"),
    validated_pixel_rate=("decomposition_usable", "mean"),
    dual_positive_pixel_rate=("dual_parent_evidence_positive", "mean"),
    validation_gate_passed=("validation_gate_passed", "all"),
)
field_dna["unrestricted_parent_a_share"] = np.nan
field_dna["raw_unrestricted_parent_a_share"] = np.nan
field_dna["off_target_mass"] = np.nan
for mixture_label, parents in MIXTURE_PARENTS.items():
    mask = field_dna["mixture_cohort"].eq(mixture_label)
    raw_parent_columns = [
        f"raw_weight_{parent.lower().replace(' ', '_')}"
        for parent in parents
    ]
    raw_parent_mass = field_dna.loc[mask, raw_parent_columns].sum(axis=1)
    field_dna.loc[mask, "raw_named_parent_mass"] = raw_parent_mass
    field_dna.loc[mask, "named_parent_mass"] = raw_parent_mass
    conditional_parent_share = np.divide(
        field_dna.loc[mask, raw_parent_columns[0]].to_numpy(np.float64),
        raw_parent_mass.to_numpy(np.float64),
        out=np.full(mask.sum(), np.nan, dtype=np.float64),
        where=raw_parent_mass.to_numpy() > 1e-12,
    )
    field_dna.loc[mask, "raw_unrestricted_parent_a_share"] = (
        conditional_parent_share
    )
    field_dna.loc[mask, "unrestricted_parent_a_share"] = (
        conditional_parent_share
    )
    field_dna.loc[mask, "off_target_mass"] = 1.0 - raw_parent_mass
    field_dna.loc[mask, "dual_parent_score"] = (
        4.0
        * field_dna.loc[mask, raw_parent_columns[0]]
        * field_dna.loc[mask, raw_parent_columns[1]]
        / raw_parent_mass.clip(lower=1e-12)
    )
field_dna["dual_parent_field_threshold"] = [
    field_dual_thresholds[(cohort, window)]
    for cohort, window in field_dna[
        ["mixture_cohort", "window_id"]
    ].itertuples(index=False, name=None)
]
field_dna["field_residual_threshold"] = [
    float(np.quantile(field_residual_controls[(cohort, window)], 0.95))
    for cohort, window in field_dna[
        ["mixture_cohort", "window_id"]
    ].itertuples(index=False, name=None)
]
field_dna["field_pair_residual_threshold"] = [
    float(np.quantile(field_pair_residual_controls[(cohort, window)], 0.95))
    for cohort, window in field_dna[
        ["mixture_cohort", "window_id"]
    ].itertuples(index=False, name=None)
]
field_dna["field_named_mass_threshold"] = [
    named_mass_thresholds[(cohort, window)]
    for cohort, window in field_dna[
        ["mixture_cohort", "window_id"]
    ].itertuples(index=False, name=None)
]
field_dna["field_quality_passed"] = (
    field_dna["usable_pixel_rate"].ge(0.5)
    & field_dna["pair_in_segment_rate"].ge(0.5)
    & field_dna["raw_named_parent_mass"].ge(
        field_dna["field_named_mass_threshold"]
    )
    & field_dna["all_parent_residual_scaled"].le(
        field_dna["field_residual_threshold"]
    )
    & field_dna["pair_residual_scaled"].le(
        field_dna["field_pair_residual_threshold"]
    )
)
field_dna["detection_eligible"] = (
    field_dna["validation_gate_passed"] & field_dna["field_quality_passed"]
)
field_dna["field_decomposition_usable"] = field_dna["detection_eligible"]
field_dna["dual_parent_score_exceeds_null"] = (
    field_dna["dual_parent_score"].gt(field_dna["dual_parent_field_threshold"])
)
field_dna["intercropping_detected"] = (
    field_dna["field_decomposition_usable"]
    & field_dna["dual_parent_score_exceeds_null"]
)
field_dna["call_status"] = np.select(
    [
        ~field_dna["validation_gate_passed"],
        ~field_dna["field_quality_passed"],
        field_dna["intercropping_detected"],
    ],
    [
        "RAW ESTIMATE — VALIDATION FAILED",
        "RAW ESTIMATE — FIELD QC FAILED",
        "VALIDATED DUAL-PARENT DETECTION",
    ],
    default="VALIDATED DECOMPOSITION; NO DUAL-PARENT DETECTION",
)

# Reference-field bootstrap: same sampled reference IDs are reused through w1-w4.
bootstrap_rng = np.random.default_rng(RANDOM_SEED + 500)
field_bootstrap_records = []
group_bootstrap_records = []
group_field_ids = {
    mixture_label: sorted(
        field_dna.loc[
            field_dna["mixture_cohort"].eq(mixture_label), "field_uid"
        ].astype(str).unique()
    )
    for mixture_label in MIXTURE_PARENTS
}
for mixture_label, parents in MIXTURE_PARENTS.items():
    mixture_fields = sorted(
        pixel_dna.loc[
            pixel_dna["mixture_cohort"].eq(mixture_label), "field_uid"
        ].unique()
    )
    paired_group_fields = group_field_ids[mixture_label]
    for bootstrap in range(N_REFERENCE_BOOTSTRAPS):
        sampled_ids_by_crop = {
            crop: bootstrap_rng.choice(
                reference_panels[mixture_label][crop],
                size=len(reference_panels[mixture_label][crop]),
                replace=True,
            )
            for crop in MONO_CROPS
        }
        sampled_mixture_fields = (
            bootstrap_rng.choice(
                paired_group_fields,
                size=len(paired_group_fields),
                replace=True,
            )
            if paired_group_fields
            else np.array([], dtype=object)
        )
        for window in WINDOWS:
            model = full_models[(mixture_label, window)]
            signatures = model["field_signatures"].set_index("field_uid")
            bootstrap_prototypes = np.stack(
                [
                    np.stack(
                        signatures.loc[
                            list(sampled_ids_by_crop[crop]), "signature"
                        ].to_numpy()
                    ).mean(axis=0)
                    for crop in MONO_CROPS
                ]
            )
            rows = timeline[
                timeline["landcover"].eq(mixture_label)
                & timeline["window_id"].eq(window)
            ]
            scored = score_embeddings(
                rows,
                model["standardizer"],
                bootstrap_prototypes,
                parents,
            )
            per_field = scored.groupby("field_uid", as_index=False).agg(
                **{column: (column, "mean") for column in weight_columns},
                pair_parent_a_share=("pair_parent_a_share", "mean"),
                named_parent_mass=("named_parent_mass", "mean"),
            )
            parent_columns = [
                f"weight_{parent.lower().replace(' ', '_')}"
                for parent in parents
            ]
            bootstrap_parent_mass = per_field[parent_columns].sum(axis=1)
            per_field["unrestricted_parent_a_share"] = np.divide(
                per_field[parent_columns[0]].to_numpy(np.float64),
                bootstrap_parent_mass.to_numpy(np.float64),
                out=np.full(len(per_field), np.nan, dtype=np.float64),
                where=bootstrap_parent_mass.to_numpy() > 1e-12,
            )
            per_field["mixture_cohort"] = mixture_label
            per_field["window_id"] = window
            per_field["bootstrap"] = bootstrap
            field_bootstrap_records.append(per_field)
            sampled_group = (
                per_field.set_index("field_uid").loc[
                    list(sampled_mixture_fields)
                ]
                if len(sampled_mixture_fields)
                else pd.DataFrame(columns=per_field.columns)
            )
            group_bootstrap_records.append(
                {
                    "mixture_cohort": mixture_label,
                    "window_id": window,
                    "bootstrap": bootstrap,
                    **{
                        column: float(sampled_group[column].mean())
                        for column in [
                            *weight_columns,
                            "unrestricted_parent_a_share",
                            "pair_parent_a_share",
                            "named_parent_mass",
                        ]
                    },
                }
            )
field_bootstrap = pd.concat(field_bootstrap_records, ignore_index=True)
group_bootstrap = pd.DataFrame(group_bootstrap_records)
bootstrap_metrics = [
    *weight_columns,
    "unrestricted_parent_a_share",
    "pair_parent_a_share",
    "named_parent_mass",
]
field_intervals = field_bootstrap.groupby(
    ["mixture_cohort", "field_uid", "window_id"]
)[bootstrap_metrics].quantile([0.025, 0.975]).unstack()
field_intervals.columns = [
    f"{metric}_{'ci_low' if quantile == 0.025 else 'ci_high'}"
    for metric, quantile in field_intervals.columns
]
field_dna = field_dna.merge(
    field_intervals.reset_index(),
    on=["mixture_cohort", "field_uid", "window_id"],
    how="left",
    validate="one_to_one",
)
field_dna["pair_share_ci_width"] = (
    field_dna["pair_parent_a_share_ci_high"]
    - field_dna["pair_parent_a_share_ci_low"]
)
# Raw field estimates and bootstrap intervals remain visible regardless of validation.

group_dna = field_dna.groupby(
    ["mixture_cohort", "window_id", "window_order"],
    as_index=False,
).agg(
    **{column: (column, "mean") for column in weight_columns},
    fields_total=("field_uid", "nunique"),
    fields_quality_passed=("field_quality_passed", "sum"),
    fields_detection_eligible=("detection_eligible", "sum"),
    fields_detected=("intercropping_detected", "sum"),
    unrestricted_parent_a_share=("unrestricted_parent_a_share", "mean"),
    pair_parent_a_share=("pair_parent_a_share", "mean"),
    named_parent_mass=("named_parent_mass", "mean"),
    off_target_mass=("off_target_mass", "mean"),
    residual_percentile=("residual_percentile", "median"),
    descriptive_dual_parent_rate=("dual_parent_score_exceeds_null", "mean"),
    overall_detection_yield=("intercropping_detected", "mean"),
)
group_dna["validated_detection_rate"] = np.divide(
    group_dna["fields_detected"],
    group_dna["fields_detection_eligible"],
    out=np.full(len(group_dna), np.nan, dtype=np.float64),
    where=group_dna["fields_detection_eligible"].to_numpy() > 0,
)

all_groups = field_dna[
    ["mixture_cohort", "window_id", "window_order"]
].drop_duplicates()
group_dna = all_groups.merge(
    group_dna,
    on=["mixture_cohort", "window_id", "window_order"],
    how="left",
    validate="one_to_one",
)
group_intervals = group_bootstrap.groupby(
    ["mixture_cohort", "window_id"]
)[[
    "unrestricted_parent_a_share",
    "pair_parent_a_share",
    "named_parent_mass",
]].quantile(
    [0.025, 0.975]
).unstack()
group_intervals.columns = [
    f"{metric}_{'ci_low' if quantile == 0.025 else 'ci_high'}"
    for metric, quantile in group_intervals.columns
]
group_dna = group_dna.merge(
    group_intervals.reset_index(),
    on=["mixture_cohort", "window_id"],
    how="left",
    validate="one_to_one",
)

group_view = group_dna.set_index(["mixture_cohort", "window_id"])
field_view = field_dna.sort_values(
    ["mixture_cohort", "field_uid", "window_order"],
    kind="stable",
).set_index(["mixture_cohort", "field_uid", "window_id"])

display(Markdown("### All-field raw decomposition — four-crop weights always populated"))
display(
    group_view[
        [
            *weight_columns,
            "unrestricted_parent_a_share",
            "unrestricted_parent_a_share_ci_low",
            "unrestricted_parent_a_share_ci_high",
            "pair_parent_a_share",
            "pair_parent_a_share_ci_low",
            "pair_parent_a_share_ci_high",
            "named_parent_mass",
            "named_parent_mass_ci_low",
            "named_parent_mass_ci_high",
            "off_target_mass",
            "fields_total",
        ]
    ]
)
display(Markdown("### Group validation and detection — separate from raw weights"))
display(
    group_view[
        [
            "residual_percentile",
            "descriptive_dual_parent_rate",
            "overall_detection_yield",
            "validated_detection_rate",
            "fields_total",
            "fields_quality_passed",
            "fields_detection_eligible",
            "fields_detected",
        ]
    ]
)
display(Markdown("### Raw field decomposition — four-crop weights always populated"))
display(
    field_view[
        [
            *weight_columns,
            "unrestricted_parent_a_share",
            "pair_parent_a_share",
            "named_parent_mass",
            "off_target_mass",
            "pixel_count",
            "call_status",
        ]
    ]
)
display(Markdown("### Field uncertainty, quality, and validated detection"))
display(
    field_view[
        [
            "unrestricted_parent_a_share_ci_low",
            "unrestricted_parent_a_share_ci_high",
            "pair_parent_a_share_ci_low",
            "pair_parent_a_share_ci_high",
            "pair_parent_a_q10",
            "pair_parent_a_q90",
            "dual_parent_score",
            "dual_parent_field_threshold",
            "residual_percentile",
            "pair_residual_percentile",
            "usable_pixel_rate",
            "validation_gate_passed",
            "field_quality_passed",
            "detection_eligible",
            "dual_parent_score_exceeds_null",
            "intercropping_detected",
        ]
    ]
)

print(
    f"Scored {pixel_dna['pixel_id'].nunique():,} exact mixture pixels and "
    f"{field_dna['field_uid'].nunique():,} mixture fields through all four windows."
)

In [ ]:
def watermark(figure, extra=None):
    messages = []
    if not pipeline_complete:
        messages.append("PARTIAL PIPELINE SNAPSHOT — DESCRIPTIVE ONLY")
    if extra:
        messages.append(str(extra))
    if messages:
        figure.text(
            0.5,
            0.5,
            " · ".join(messages),
            ha="center",
            va="center",
            fontsize=18,
            color="crimson",
            alpha=0.16,
            rotation=24,
            weight="bold",
        )


example = (
    pixel_dna[
        pixel_dna["mixture_cohort"].eq("Bean and Maize")
        & pixel_dna["window_id"].eq("w4")
    ]
    .sort_values(
        ["pair_in_segment", "named_parent_mass", "residual_percentile"],
        ascending=[False, False, True],
        kind="stable",
    )
    .iloc[0]
)
example_model = full_models[(example["mixture_cohort"], example["window_id"])]
example_control_source = validation_table.loc[
    (validation_table["mixture"] == example["mixture_cohort"])
    & (validation_table["window_id"] == example["window_id"]),
    "control_source",
].iloc[0].replace("_", " ")
target = np.asarray(example["_standardized"])
prototypes = example_model["prototypes"]
simplex = solve_simplex_admixture(
    target,
    prototypes,
    prototype_names=MONO_CROPS,
)
heat_rows = np.vstack(
    [
        target,
        *prototypes,
        simplex.reconstruction[0],
        simplex.residual[0],
    ]
)
heat_labels = [
    "observed intercropped pixel",
    *[f"{crop} prototype" for crop in MONO_CROPS],
    "fitted 4-crop reconstruction",
    "unexplained residual vector",
]
color_limit = max(float(np.quantile(np.abs(heat_rows), 0.98)), 1e-6)

figure = plt.figure(figsize=(17, 8))
grid = figure.add_gridspec(2, 2, height_ratios=[1.35, 1.0])
heat_axis = figure.add_subplot(grid[0, :])
image = heat_axis.imshow(
    heat_rows,
    cmap="coolwarm",
    norm=TwoSlopeNorm(vmin=-color_limit, vcenter=0.0, vmax=color_limit),
    aspect="auto",
)
heat_axis.set_yticks(range(len(heat_labels)), heat_labels)
heat_axis.set_xticks(range(0, 128, 8), [str(index + 1) for index in range(0, 128, 8)])
heat_axis.set_xlabel("TESSERA embedding dimension")
heat_axis.set_title(
    "The DNA match uses all 128 numbers: observed ≈ weighted monocrop prototypes + residual"
)
figure.colorbar(image, ax=heat_axis, fraction=0.018, pad=0.015, label="fixed standardized value")

weight_axis = figure.add_subplot(grid[1, 0])
left = 0.0
for crop, weight in zip(MONO_CROPS, simplex.weights[0], strict=True):
    weight_axis.barh(
        [0],
        [100.0 * weight],
        left=left,
        color=CROP_COLORS[crop],
        label=f"{crop}: {weight:.1%}",
    )
    left += 100.0 * weight
weight_axis.set_xlim(0, 100)
weight_axis.set_yticks([])
weight_axis.set_xlabel("4-crop embedding-signature share (%)")
weight_axis.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.18))

card_axis = figure.add_subplot(grid[1, 1])
card_axis.axis("off")
card_axis.text(
    0.02,
    0.95,
    "Named-parent decomposition",
    fontsize=14,
    weight="bold",
    va="top",
)
card_axis.text(
    0.02,
    0.76,
    f"Bean signature within Bean–Maize segment: {example['pair_parent_a_share']:.1%}\n"
    f"Maize signature within Bean–Maize segment: {1-example['pair_parent_a_share']:.1%}\n"
    f"Named-parent mass in unrestricted fit: {example['named_parent_mass']:.1%}\n"
    f"Off-target mass: {example['off_target_mass']:.1%}\n"
    f"Scaled residual: {example['all_parent_residual_scaled']:.3f}× reference radius\n"
        f"Residual percentile ({example_control_source}): "
        f"{example['residual_percentile']:.1f}%\n"
    f"Projection inside parent segment: {bool(example['pair_in_segment'])}\n"
        f"Decomposition status: {example['call_status']}",
    fontsize=12,
    va="top",
    linespacing=1.45,
)
card_axis.text(
    0.02,
    0.05,
    "These percentages describe the fitted embedding signature—not planted area or biomass.",
    color="crimson",
    fontsize=10,
    weight="bold",
)
figure.suptitle(
    f"Example field={example['field_uid']} · pixel={example['pixel_id']} · w4",
    fontsize=14,
)
watermark(
    figure,
    None
    if bool(example["validation_gate_passed"])
    else "RAW DESCRIPTIVE EXAMPLE — VALIDATION FAILED",
)
figure.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(
    len(MIXTURE_PARENTS),
    len(WINDOWS),
    figsize=(16, 7.5),
    sharex=True,
    sharey=True,
)
for row, mixture_label in enumerate(MIXTURE_PARENTS):
    for column, window in enumerate(WINDOWS):
        axis = axes[row, column]
        matrix = validation_matrices[(mixture_label, window)]
        image = axis.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
        axis.set_xticks(range(len(MONO_CROPS)), MONO_CROPS, rotation=45, ha="right", fontsize=8)
        axis.set_yticks(range(len(MONO_CROPS)), MONO_CROPS, fontsize=8)
        for actual in range(len(MONO_CROPS)):
            for recovered in range(len(MONO_CROPS)):
                axis.text(
                    recovered,
                    actual,
                    f"{matrix[actual, recovered]:.0%}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if matrix[actual, recovered] > 0.55 else "black",
                )
        metrics = validation_table[
            validation_table["mixture"].eq(mixture_label)
            & validation_table["window_id"].eq(window)
        ].iloc[0]
        gate_text = "PASS" if metrics["validation_gate_passed"] else "NO-CALL"
        source_text = (
            "5 km grid-block OOF"
            if metrics["control_source"] == "grid_block_oof"
            else "in-sample fallback"
        )
        axis.set_title(
            f"{window} · {gate_text} · {source_text}\n"
            f"BA={metrics['balanced_accuracy']:.2f}, endpoint MAE={metrics['endpoint_mae']:.2f}",
            color="darkgreen" if gate_text == "PASS" else "crimson",
            fontsize=9,
        )
        if column == 0:
            axis.set_ylabel(f"{mixture_label}\ntrue monocrop control")
        if row == len(MIXTURE_PARENTS) - 1:
            axis.set_xlabel("recovered embedding-signature component")
color_axis = figure.add_axes([0.93, 0.21, 0.012, 0.56])
figure.colorbar(image, cax=color_axis)
figure.suptitle(
    "Monocrop controls: held out by 5 km grid block where blocks permit; otherwise raw-only with no validated detection",
    fontsize=14,
)
watermark(figure)
figure.subplots_adjust(
    left=0.08,
    right=0.91,
    top=0.84,
    bottom=0.20,
    wspace=0.18,
    hspace=0.32,
)
plt.show()
display(validation_table.set_index(["mixture", "window_id"]))

In [ ]:
for mixture_label in MIXTURE_PARENTS:
    rows = field_dna[
        field_dna["mixture_cohort"].eq(mixture_label)
        & field_dna["window_id"].eq("w4")
    ].sort_values("pair_parent_a_share", kind="stable")
    height = max(6.0, 0.28 * len(rows) + 2.0)
    figure, (weight_axis, residual_axis) = plt.subplots(
        1,
        2,
        figsize=(15, height),
        gridspec_kw={"width_ratios": [4.5, 1.2]},
        sharey=True,
    )
    y = np.arange(len(rows))
    left = np.zeros(len(rows))
    for crop, column in zip(MONO_CROPS, weight_columns, strict=True):
        values = 100.0 * rows[column].to_numpy()
        weight_axis.barh(
            y,
            values,
            left=left,
            color=CROP_COLORS[crop],
            label=crop,
        )
        left += values
    weight_axis.set_xlim(0, 100)
    weight_axis.set_xlabel("4-crop embedding-signature share (%)")
    weight_axis.set_yticks(
        y,
        [
            f"{field_uid[:12]}… ({pixel_count} px)"
            for field_uid, pixel_count in rows[
                ["field_uid", "pixel_count"]
            ].itertuples(index=False, name=None)
        ],
        fontsize=7,
    )
    no_call = ~rows["field_decomposition_usable"].to_numpy(bool)
    if no_call.any():
        weight_axis.barh(
            y[no_call], 100, facecolor="none", edgecolor="crimson", linewidth=1.0,
            label="RAW / NOT VALIDATED",
        )
    valid_intervals = rows["pair_parent_a_share"].notna()
    if valid_intervals.any():
        centers = 100.0 * rows.loc[valid_intervals, "pair_parent_a_share"]
        lower = 100.0 * (
            rows.loc[valid_intervals, "pair_parent_a_share"]
            - rows.loc[valid_intervals, "pair_parent_a_q10"]
        )
        upper = 100.0 * (
            rows.loc[valid_intervals, "pair_parent_a_q90"]
            - rows.loc[valid_intervals, "pair_parent_a_share"]
        )
        weight_axis.errorbar(
            centers, y[valid_intervals.to_numpy()], xerr=np.vstack([lower, upper]),
            fmt="D", color="black", markersize=3, linewidth=0.8,
            label="named-pair share; pixel q10–q90",
        )
    bootstrap_intervals = (
        rows["pair_parent_a_share"].notna()
        & rows["pair_parent_a_share_ci_low"].notna()
        & rows["pair_parent_a_share_ci_high"].notna()
    )
    if bootstrap_intervals.any():
        centers = 100.0 * rows.loc[
            bootstrap_intervals, "pair_parent_a_share"
        ]
        lower = 100.0 * (
            rows.loc[bootstrap_intervals, "pair_parent_a_share"]
            - rows.loc[bootstrap_intervals, "pair_parent_a_share_ci_low"]
        ).clip(lower=0)
        upper = 100.0 * (
            rows.loc[bootstrap_intervals, "pair_parent_a_share_ci_high"]
            - rows.loc[bootstrap_intervals, "pair_parent_a_share"]
        ).clip(lower=0)
        weight_axis.errorbar(
            centers,
            y[bootstrap_intervals.to_numpy()],
            xerr=np.vstack([lower, upper]),
            fmt="o",
            markerfacecolor="white",
            markeredgecolor="#0B4F6C",
            color="#0B4F6C",
            markersize=4,
            linewidth=1.0,
            capsize=2,
            label="95% reference-field bootstrap",
        )
    weight_axis.legend(
        ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.01), fontsize=7.5
    )
    weight_axis.grid(axis="x", alpha=0.15)

    residual_scatter = residual_axis.scatter(
        rows["residual_percentile"],
        y,
        c=rows["named_parent_mass"],
        cmap="viridis",
        vmin=0,
        vmax=1,
        s=32,
    )
    residual_axis.axvline(95, color="crimson", linestyle="--", linewidth=1)
    residual_axis.set_xlim(0, 100)
    control_source = validation_table.loc[
        (validation_table["mixture"] == mixture_label)
        & (validation_table["window_id"] == "w4"),
        "control_source",
    ].iloc[0].replace("_", " ")
    residual_axis.set_xlabel(
        f"scaled control residual percentile ({control_source})\n"
        "higher = worse fit"
    )
    residual_axis.grid(axis="x", alpha=0.15)
    figure.colorbar(
        residual_scatter, ax=residual_axis, fraction=0.08, pad=0.04,
        label="named-parent mass",
    )
    figure.suptitle(
        f"{mixture_label} · w4 DNA decomposition · {RUN_STATUS}\n"
        "Residual dots are colored by named-parent mass",
        color="black" if pipeline_complete else "crimson",
    )
    watermark(
        figure,
        None
        if validation_gates[(mixture_label, "w4")]
        else "VALIDATION FAILED — RAW FIELD SIGNATURES SHOWN",
    )
    figure.tight_layout()
    plt.show()

figure, axes = plt.subplots(
    len(MIXTURE_PARENTS),
    1,
    figsize=(12, 7),
    sharex=True,
)
for axis, mixture_label in zip(axes, MIXTURE_PARENTS, strict=True):
    rows = group_dna[group_dna["mixture_cohort"].eq(mixture_label)].sort_values(
        "window_order"
    )
    x = np.arange(len(WINDOWS))
    bottom = np.zeros(len(rows))
    for crop, column in zip(MONO_CROPS, weight_columns, strict=True):
        values = 100.0 * rows[column].to_numpy()
        axis.bar(x, values, bottom=bottom, color=CROP_COLORS[crop], label=crop)
        bottom += values
    axis.set_ylabel(f"{mixture_label}\nsignature share (%)")
    axis.set_ylim(0, 100)
    axis.axvspan(2.5, 3.5, color="0.85", alpha=0.5, hatch="//")
    axis.grid(axis="y", alpha=0.15)
    for position, window in enumerate(WINDOWS):
        row = rows.iloc[position]
        axis.text(
            position,
            2,
            f"{int(row['fields_detection_eligible'])}/{int(row['fields_total'])} "
            "detection eligible",
            ha="center",
            va="bottom",
            rotation=0,
            color="white",
            fontsize=6.5,
            weight="bold",
        )
        if not validation_gates[(mixture_label, window)]:
            axis.text(position, 98, "RAW · VALIDATION FAILED", ha="center", va="top",
                      rotation=90, color="crimson", weight="bold", fontsize=8)
axes[-1].set_xticks(range(len(WINDOWS)), WINDOWS)
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 0.01))
figure.suptitle(
    "All-field raw monocrop DNA composition through cumulative windows\n"
    "w4 is hatched because it is outside TESSERA's annual contract",
    y=1.02,
)
watermark(
    figure,
    "VALIDATION FAILED IN ONE OR MORE WINDOWS — RAW SIGNATURES SHOWN"
    if not all(validation_gates.values()) else None,
)
figure.tight_layout(rect=[0, 0.08, 1, 0.92])
plt.show()

In [ ]:
def plot_boundary(axis, field_uid, epsg):
    geometry_text = static.field_lookup.loc[field_uid, "wkt"]
    geometry = shapely_wkt.loads(str(geometry_text))
    transformer = Transformer.from_crs(4326, int(epsg), always_xy=True)
    projected = shapely_transform(transformer.transform, geometry)
    geometries = list(projected.geoms) if projected.geom_type == "MultiPolygon" else [projected]
    for polygon in geometries:
        x, y = polygon.exterior.xy
        axis.plot(np.asarray(x) / 10.0, np.asarray(y) / 10.0, color="black", linewidth=1)
        for interior in polygon.interiors:
            x, y = interior.xy
            axis.plot(np.asarray(x) / 10.0, np.asarray(y) / 10.0, color="black", linewidth=0.7)


map_fields = []
for mixture_label in MIXTURE_PARENTS:
    candidates = (
        field_dna[field_dna["mixture_cohort"].eq(mixture_label)]
        .groupby("field_uid", as_index=False)["pixel_count"]
        .max()
        .sort_values(["pixel_count", "field_uid"], ascending=[False, True])
        .head(PIXEL_MAP_FIELDS_PER_MIXTURE)
    )
    map_fields.extend(candidates["field_uid"].tolist())

figure, axes = plt.subplots(
    len(map_fields),
    len(WINDOWS),
    figsize=(16, 3.6 * len(map_fields)),
    squeeze=False,
)
for row, field_uid in enumerate(map_fields):
    field_rows = pixel_dna[pixel_dna["field_uid"].eq(field_uid)]
    mixture_label = str(field_rows["mixture_cohort"].iloc[0])
    parent_a, parent_b = MIXTURE_PARENTS[mixture_label]
    for column, window in enumerate(WINDOWS):
        axis = axes[row, column]
        rows = field_rows[field_rows["window_id"].eq(window)]
        color_map = plt.get_cmap("coolwarm")
        color_norm = Normalize(vmin=0, vmax=1)
        for pixel in rows.itertuples(index=False):
            facecolor = color_map(color_norm(pixel.pair_parent_a_share))
            hatch = None if bool(pixel.fit_usable) else "///"
            axis.add_patch(
                Rectangle(
                    (pixel.pixel_x_index, pixel.pixel_y_index),
                    1.0,
                    1.0,
                    facecolor=facecolor,
                    edgecolor="black",
                    linewidth=0.35,
                    hatch=hatch,
                )
            )
        axis.autoscale_view()
        scatter = plt.cm.ScalarMappable(norm=color_norm, cmap=color_map)
        plot_boundary(axis, field_uid, int(rows["utm_epsg"].iloc[0]))
        axis.set_aspect("equal")
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_title(
            f"{window}{' · OOD' if window == 'w4' else ''}"
            f"{' · RAW/UNVALIDATED' if not validation_gates[(mixture_label, window)] else ''}\n"
            f"{parent_b} ← share → {parent_a}",
            fontsize=9,
        )
        if column == 0:
            axis.set_ylabel(
                f"{mixture_label}\n{field_uid[:16]}…\n{rows['pixel_id'].nunique()} pixels",
                fontsize=8,
            )
figure.subplots_adjust(right=0.90, top=0.94, hspace=0.25, wspace=0.10)
color_axis = figure.add_axes([0.92, 0.15, 0.015, 0.70])
colorbar = figure.colorbar(scatter, cax=color_axis)
colorbar.set_label("named-pair first-parent embedding-signature share")
figure.suptitle(
    "Actual 10 m field pixels: how the same pixels' parent balance evolves",
    fontsize=14,
)
watermark(figure)
plt.show()

In [ ]:
metric_specs = [
    ("pair_parent_a_share", "named-pair first-parent share", (0, 1)),
    ("named_parent_mass", "named-parent mass in 4-crop fit", (0, 1)),
    (
        "residual_percentile",
        "scaled control residual percentile (OOF or fallback)",
        (0, 100),
    ),
]
figure, axes = plt.subplots(
    len(MIXTURE_PARENTS),
    len(metric_specs),
    figsize=(17, 8),
    sharex=True,
)
for row, (mixture_label, parents) in enumerate(MIXTURE_PARENTS.items()):
    rows = field_dna[field_dna["mixture_cohort"].eq(mixture_label)]
    for column, (metric, title, limits) in enumerate(metric_specs):
        axis = axes[row, column]
        for _, field_rows in rows.groupby("field_uid", sort=False):
            field_rows = field_rows.sort_values("window_order")
            axis.plot(
                field_rows["window_order"],
                field_rows[metric],
                color="0.65",
                alpha=0.28,
                linewidth=0.8,
            )
        summary = rows.groupby("window_order")[metric].agg(
            median="median",
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
        ).reindex(range(len(WINDOWS)))
        x = np.arange(len(WINDOWS))
        axis.fill_between(
            x,
            summary["q25"],
            summary["q75"],
            alpha=0.22,
            color="#1F77B4",
        )
        axis.plot(
            x,
            summary["median"],
            marker="o",
            color="#1F77B4",
            linewidth=2.2,
        )
        axis.axvspan(2.5, 3.5, color="0.85", alpha=0.5, hatch="//")
        if metric == "residual_percentile":
            axis.axhline(95, color="crimson", linestyle="--", linewidth=1)
        axis.set_ylim(*limits)
        axis.set_title(title)
        axis.grid(alpha=0.15)
        if column == 0:
            axis.set_ylabel(
                f"{mixture_label}\n0={parents[1]}, 1={parents[0]}"
            )
        axis.set_xticks(range(len(WINDOWS)), WINDOWS)
figure.suptitle(
    "One line per field; thick line = median, band = interquartile range\n"
    "Raw field signatures remain visible; validation status is shown separately",
    fontsize=14,
)
watermark(
    figure,
    "VALIDATION FAILED IN ONE OR MORE WINDOWS — RAW SIGNATURES SHOWN"
    if not all(validation_gates.values()) else None,
)
figure.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

figure, axes = plt.subplots(
    len(MIXTURE_PARENTS),
    3,
    figsize=(14, 8),
    sharex=True,
)
for row, mixture_label in enumerate(MIXTURE_PARENTS):
    rows = pixel_dna[pixel_dna["mixture_cohort"].eq(mixture_label)].copy()
    row_index = rows[["field_uid", "pixel_id"]].drop_duplicates().sort_values(
        ["field_uid", "pixel_id"], kind="stable"
    )
    index = pd.MultiIndex.from_frame(row_index)
    heat_specs = [
        ("pair_parent_a_share", "parent balance", "coolwarm", 0, 1),
        ("named_parent_mass", "named-parent mass", "viridis", 0, 1),
        ("residual_percentile", "residual percentile", "magma", 0, 100),
    ]
    for column, (metric, title, cmap, vmin, vmax) in enumerate(heat_specs):
        usable = pd.Series(True, index=rows.index)
        matrix = (
            rows.assign(_heat_value=rows[metric].where(usable))
            .pivot_table(
                index=["field_uid", "pixel_id"],
                columns="window_id",
                values="_heat_value",
                aggfunc="first",
                dropna=False,
            )
            .reindex(index=index, columns=WINDOWS)
        )
        heat_cmap = plt.get_cmap(cmap).copy()
        heat_cmap.set_bad("0.85")
        image = axes[row, column].imshow(
            matrix.to_numpy(),
            cmap=heat_cmap,
            vmin=vmin,
            vmax=vmax,
            aspect="auto",
        )
        axes[row, column].set_title(title)
        axes[row, column].set_xticks(range(len(WINDOWS)), WINDOWS)
        axes[row, column].set_yticks([])
        if column == 0:
            axes[row, column].set_ylabel(
                f"{mixture_label}\n{len(matrix)} exact pixels"
            )
        figure.colorbar(
            image,
            ax=axes[row, column],
            fraction=0.046,
            pad=0.03,
        )
figure.suptitle(
    "Same-pixel raw DNA evolution (validation status is separate)",
    fontsize=14,
)
watermark(
    figure,
    "VALIDATION FAILED IN ONE OR MORE WINDOWS — RAW SIGNATURES SHOWN"
    if not all(validation_gates.values()) else None,
)
figure.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

clarity = validation_table.copy()
figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
clarity_metrics = [
    ("balanced_accuracy", "monocrop control balanced accuracy", (0, 1)),
    ("pair_separation", "parent separation / within-parent spread", None),
    ("endpoint_mae", "monocrop endpoint share MAE (lower is better)", (0, 1)),
]
for axis, (metric, title, limits) in zip(axes, clarity_metrics, strict=True):
    for mixture_label in MIXTURE_PARENTS:
        rows = clarity[clarity["mixture"].eq(mixture_label)].copy()
        rows["window_order"] = rows["window_id"].map(
            {window: index for index, window in enumerate(WINDOWS)}
        )
        rows = rows.sort_values("window_order")
        gate_mask = rows["validation_gate_passed"].to_numpy(bool)
        axis.plot(
            rows["window_order"],
            rows[metric],
            color="0.6",
            linewidth=1.0,
            label=mixture_label,
        )
        axis.scatter(
            rows["window_order"],
            rows[metric],
            c=np.where(gate_mask, "#1F77B4", "crimson"),
            marker="o",
            zorder=3,
        )
    axis.axvspan(2.5, 3.5, color="0.85", alpha=0.5, hatch="//")
    axis.set_xticks(range(len(WINDOWS)), WINDOWS)
    axis.set_title(title)
    if limits is not None:
        axis.set_ylim(*limits)
    if metric == "pair_separation":
        axis.set_ylim(bottom=0)
    axis.grid(alpha=0.15)
axes[0].legend(fontsize=8)
figure.suptitle(
    "Does incremental context improve separation? Red points = validation NO-CALL"
)
watermark(
    figure,
    "RED POINTS ARE DIAGNOSTIC ONLY"
    if not all(validation_gates.values()) else None,
)
figure.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## Interpretation guardrails

- The unrestricted four-crop simplex is the primary DNA-style decomposition. Its nonnegative weights sum to one and explicitly expose off-target monocrop mass.
- The named-parent-only ratio is a second, forced projection onto the Bean–Maize or Irish-Potato–Maize segment. Its raw value is always shown; validated interpretation is unavailable when named-parent mass is low, the projection lies outside the segment, or the fit fails its controls.
- A usable decomposition and an intercropping detection are separate. Field-level dual-parent evidence is `4 × w_parent_A × w_parent_B / (w_parent_A + w_parent_B)`: zero at a pure-parent endpoint and largest when both parents contribute equally with high named-parent mass.
- Field percentages, field detection, and bootstrap intervals all use the same raw set of exact pixels. Quality checks are applied once to the resulting field aggregate. Raw estimates and intervals remain visible; validation status is reported separately and only controls whether a calibrated detection is claimed. Pixel-level fit masks are retained only for the diagnostic pixel maps.
- The dual-parent threshold is the empirical 95th percentile of all monocrop control fields. A call also requires at least 20 independent null fields, sufficient named-parent mass, acceptable field residuals, and stable prototype geometry.
- Fold residuals are divided by each fold model's equal-crop/equal-field training-reference radius before pooling. Validation refits the metric and prototypes by 5 km grid block when possible; this is a grid-block holdout, not a guaranteed 5 km exclusion buffer. An in-sample fallback is diagnostic only and prevents a validated detection, but it does not hide the raw decomposition.
- All clean reference fields and exact physical pixels are used through w1–w4; geography is reported only as a diagnostic and does not select a subset. The full-model affine metric is fitted once from w1–w3 references; window-specific prototypes remain field-balanced. A 200-resample reference-field bootstrap quantifies prototype-selection uncertainty.
- `w4` spans 487 days and is outside TESSERA's annual temporal contract, so it is shown as an out-of-contract sensitivity window.
- These are **embedding-signature shares**, not planted-area, plant-count, biomass, yield, or physical crop fractions. Known-ratio mixture fields would be required to calibrate those physical quantities.